# 02. 추세추종 백테스트 — 내 매매 방식을 규칙으로

**질문:** 내가 보는 지표(월봉·주봉 추세, MACD, 거래량, OBV, 볼린저밴드)로 사고팔면, 그냥 들고 있는 것보다 나은가?
리스크 규칙을 더하면 낙폭이 얼마나 줄어드나?

**층을 하나씩 더해가며 비교** (각 층이 성과를 개선하는지 확인)

| 전략 | 진입 조건 | 청산 조건 | 리스크 규칙 |
|---|---|---|---|
| L0 Hold | 유니버스 동일가중 보유 (비교 기준) | – | – |
| L1 Trend | 월봉 10MA 위 + 주봉 MACD>0 + 일봉 MACD 골든크로스 | 일봉 MACD 데드크로스 또는 주봉 추세 이탈 | – |
| L2 +Vol/OBV | L1 + 거래량 > 20일 평균 + OBV > OBV 20일 평균 | 동일 | – |
| L3 +Bollinger | L2 + 종가가 볼린저 상단 근처(밴드 위치 ≥ 0.8) | 동일 | – |
| L4 +Risk | L2 와 동일 | 동일 + ATR 손절 | ATR 손절 · 변동성 비중조절 · 낙폭 방어 · 이벤트 전후 진입 금지 |

**규칙:** 15일마다(매월 1일·16일 근처 거래일) 종가로 신호 판단 → **다음 날 시가**에 매매 (미래 정보 사용 방지).
손절은 매일 체크. 유니버스는 **보험주** / **비보험 대형주(해마다 시총 상위 30, 흑자)** 두 개로 따로 돌림.

> 기본 전략(01 노트북)은 그대로 두고, 여기서 결과를 보며 규칙을 고쳐나가기. 고친 내용은 `experiments.md`에 기록.


## 0. 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pykrx import stock

START, END = "20160101", "20251231"     # 백테스트 기간
WARMUP = "20140101"                     # 지표 계산용 과거 데이터 시작
TOP_N = 30                              # 비보험 유니버스: 해마다 시총 상위 30
INSURERS = ["000810", "005830", "001450", "000370", "000400", "000540",
            "032830", "088350", "085620", "082640", "003690"]

# 매매 비용 (수수료 + 슬리피지, 매도는 거래세 포함 근사)
COST_BUY, COST_SELL = 0.0015, 0.0035
RF = 0.03

# 지표 파라미터 — 결과 보면서 바꿔보기
MACD_FAST, MACD_SLOW, MACD_SIG = 12, 26, 9
MONTH_MA = 10            # 월봉 10개월 이동평균
VOL_MA = OBV_MA = BB_N = 20
BB_K = 2.0
BB_POS_MIN = 0.8         # L3: 볼린저 밴드 안 위치(0=하단, 1=상단) 이 값 이상일 때만 진입

# 리스크 파라미터 (L4)
ATR_N, ATR_MULT = 14, 2.0     # 손절선 = 진입가 - 2×ATR
RISK_PER_TRADE = 0.01         # 한 번 손절 시 계좌의 1%만 잃도록 비중 결정
MAX_WEIGHT = 0.20             # 종목당 최대 20%
DD_GUARD, DD_SCALE = -0.15, 0.5   # 계좌 낙폭 -15% 넘으면 신규 비중 절반
EVENT_BLOCK_DAYS = 3          # 이벤트 ±3일 신규 진입 금지

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 1. 데이터 받기 (처음 한 번 10~20분, 이후 캐시)

In [ ]:
def load_cached(name, fn):
    path = f"data/{name}.pkl"
    if os.path.exists(path):
        return pd.read_pickle(path)
    df = fn()
    df.to_pickle(path)
    time.sleep(0.5)
    return df

# 해마다 1월 첫 거래일 기준 유니버스 (그 해 동안 매매 가능 종목)
years = range(int(START[:4]), int(END[:4]) + 1)
yearly_uni = {}
for y in years:
    d = stock.get_nearest_business_day_in_a_week(f"{y}0102", prev=False)
    cap = load_cached(f"cap_{d}", lambda: stock.get_market_cap(d, market="KOSPI"))
    fnd = load_cached(f"fund_{d}", lambda: stock.get_market_fundamental(d, market="KOSPI"))
    profitable = set(fnd.index[fnd["PER"] > 0])
    big = [t for t in cap.drop(INSURERS, errors="ignore")["시가총액"].sort_values(ascending=False).index
           if t in profitable][:TOP_N]
    ins = [t for t in INSURERS if t in cap.index]
    yearly_uni[y] = {"NonIns": big, "Ins": ins}

all_tickers = sorted({t for u in yearly_uni.values() for g in u.values() for t in g})
print("일봉 받을 종목 수:", len(all_tickers))

ohlcv = {}
for i, t in enumerate(all_tickers):
    ohlcv[t] = load_cached(f"ohlcv_{t}", lambda: stock.get_market_ohlcv(WARMUP, END, t))
    if i % 10 == 0: print(i, t)

kospi = stock.get_index_ohlcv(WARMUP, END, "1001")["종가"]
events = pd.read_csv("../data_events/events.csv", parse_dates=["date"])

## 2. 지표 계산 — 내가 차트에서 보는 것들

- **월봉 추세**: 월말 종가 > 10개월 이동평균 (지난달까지 완성된 봉 기준)
- **주봉 MACD**: 주봉 MACD 히스토그램 > 0
- **일봉 MACD**: MACD선이 시그널선을 상향 돌파(골든크로스) / 하향 돌파(데드크로스)
- **거래량**: 당일 거래량 > 20일 평균
- **OBV**: OBV > OBV 20일 평균 (매집 흐름)
- **볼린저 위치**: (종가 − 하단) / (상단 − 하단)
- **ATR**: 14일 평균 실제 변동폭 (손절·비중에 사용)

In [ ]:
def macd(s):
    line = s.ewm(span=MACD_FAST, adjust=False).mean() - s.ewm(span=MACD_SLOW, adjust=False).mean()
    sig = line.ewm(span=MACD_SIG, adjust=False).mean()
    return line, sig

def indicators(df):
    df = df[df["종가"] > 0].copy()
    c, h, l, v = df["종가"], df["고가"], df["저가"], df["거래량"]
    out = pd.DataFrame(index=df.index)
    out["open"], out["high"], out["low"], out["close"] = df["시가"], h, l, c

    # 월봉 추세 (as-of: 그 날까지 완성된 마지막 월봉)
    m = c.resample("ME").last()
    m_up = (m > m.rolling(MONTH_MA).mean()).astype(float)
    out["month_up"] = m_up.reindex(out.index, method="ffill").fillna(0).astype(bool)

    # 주봉 MACD
    w = c.resample("W-FRI").last()
    wl, ws = macd(w)
    out["week_up"] = ((wl - ws) > 0).astype(float).reindex(out.index, method="ffill").fillna(0).astype(bool)

    # 일봉 MACD
    dl, ds = macd(c)
    out["macd_above"] = dl > ds
    out["golden"] = out["macd_above"] & ~out["macd_above"].shift(1, fill_value=False)

    # 거래량, OBV
    out["vol_ok"] = v > v.rolling(VOL_MA).mean()
    obv = (np.sign(c.diff()).fillna(0) * v).cumsum()
    out["obv_ok"] = obv > obv.rolling(OBV_MA).mean()

    # 볼린저
    mid, sd = c.rolling(BB_N).mean(), c.rolling(BB_N).std()
    out["bb_pos"] = (c - (mid - BB_K*sd)) / (2*BB_K*sd)

    # ATR
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    out["atr"] = tr.rolling(ATR_N).mean()
    return out

ind = {t: indicators(df) for t, df in ohlcv.items() if len(df) > 300}
print("지표 계산 완료:", len(ind), "종목")

## 3. 15일마다 판단하는 날짜 & 이벤트 차단일

In [ ]:
days = kospi.loc[START:END].index
decision_days = set()
for m in pd.date_range(START, END, freq="MS"):
    for target in (m, m + pd.Timedelta(days=15)):
        after = days[days >= target]
        if len(after): decision_days.add(after[0])

def event_blocked(day):
    return ((events["date"] - day).abs() <= pd.Timedelta(days=EVENT_BLOCK_DAYS)).any()

print("판단일 수:", len(decision_days))

## 4. 백테스트 엔진 — 판단은 종가, 매매는 다음 날 시가, 손절은 매일

In [ ]:
def entry_signal(x, prev_day, day, layer):
    """x: 한 종목 지표. prev_day~day 사이 골든크로스가 있었고 지금도 MACD 위에 있으면 신호"""
    win = x.loc[(x.index > prev_day) & (x.index <= day)]
    if win.empty or day not in x.index: return False
    r = x.loc[day]
    ok = win["golden"].any() and r["macd_above"] and r["month_up"] and r["week_up"]
    if layer in ("L2", "L3", "L4"):
        ok = ok and r["vol_ok"] and r["obv_ok"]
    if layer == "L3":
        ok = ok and r["bb_pos"] >= BB_POS_MIN
    return bool(ok)

def exit_signal(x, day):
    if day not in x.index: return False
    r = x.loc[day]
    return (not r["macd_above"]) or (not r["week_up"])

def run(group, layer, max_pos):
    cash, pos, trades = 1.0, {}, []
    equity, peak = [], 1.0
    pending_buy, pending_sell = [], []
    prev_dec = days[0] - pd.Timedelta(days=15)

    def px(t, day, col):
        x = ind.get(t)
        return x.at[day, col] if x is not None and day in x.index else np.nan

    def sell(t, day, price, reason):
        nonlocal cash
        p = pos.pop(t)
        cash += p["sh"] * price * (1 - COST_SELL)
        ret = price * (1 - COST_SELL) / (p["entry"] * (1 + COST_BUY)) - 1
        trades.append(dict(ticker=t, entry_day=p["day"], exit_day=day, ret=ret,
                           days=(day - p["day"]).days, reason=reason))

    for day in days:
        uni = yearly_uni[day.year][group]

        # (1) 전날 결정한 주문을 오늘 시가에 체결
        for t in pending_sell:
            if t in pos:
                o = px(t, day, "open")
                sell(t, day, o if o == o else pos[t]["last"], "signal")
        port_val = cash + sum(p["sh"] * p["last"] for p in pos.values())
        for t, w in pending_buy:
            o = px(t, day, "open")
            if t in pos or not (o == o) or o <= 0: continue
            amt = min(w * port_val, cash)
            if amt <= port_val * 0.01: continue
            sh = amt / (o * (1 + COST_BUY))
            cash -= amt
            stop = o - ATR_MULT * px(t, day, "atr") if layer == "L4" else -np.inf
            pos[t] = dict(sh=sh, entry=o, day=day, stop=stop, last=o)
        pending_buy, pending_sell = [], []

        # (2) 손절 (L4) — 장중 저가가 손절선 아래면 손절선(갭하락이면 시가)에 매도
        for t in list(pos):
            lo, o = px(t, day, "low"), px(t, day, "open")
            if lo == lo and lo <= pos[t]["stop"]:
                sell(t, day, min(o, pos[t]["stop"]) if o == o else pos[t]["stop"], "stop")

        # (3) 종가 평가 (데이터 끊긴 종목은 마지막 가격)
        for t, p in pos.items():
            c = px(t, day, "close")
            if c == c: p["last"] = c
        val = cash + sum(p["sh"] * p["last"] for p in pos.values())
        equity.append((day, val))
        peak = max(peak, val)

        # (4) 판단일이면 내일 주문 결정
        if day in decision_days:
            for t in list(pos):
                x = ind.get(t)
                if t not in uni or x is None or exit_signal(x, day):
                    pending_sell.append(t)
            slots = max_pos - (len(pos) - len(pending_sell))
            if slots > 0 and not (layer == "L4" and event_blocked(day)):
                cands = [t for t in uni if t in ind and t not in pos and entry_signal(ind[t], prev_dec, day, layer)]
                # 신호가 많으면 최근 1년 수익률(모멘텀) 높은 순
                mom = {t: ind[t]["close"].loc[:day].pct_change(250).iloc[-1] for t in cands}
                for t in sorted(cands, key=lambda t: -np.nan_to_num(mom[t], nan=-9))[:slots]:
                    if layer == "L4":
                        stop_dist = ATR_MULT * ind[t].at[day, "atr"] / ind[t].at[day, "close"]
                        w = min(RISK_PER_TRADE / stop_dist, MAX_WEIGHT) if stop_dist > 0 else 0
                        if val / peak - 1 <= DD_GUARD: w *= DD_SCALE
                    else:
                        w = 1 / max_pos
                    pending_buy.append((t, w))
            prev_dec = day

    eq = pd.Series(dict(equity))
    return eq, pd.DataFrame(trades)

def run_hold(group):
    """L0: 그 해 유니버스를 1월에 동일가중으로 사서 1년 보유"""
    val, vals = 1.0, []
    for y in years:
        uni = [t for t in yearly_uni[y][group] if t in ind]
        d = days[days.year == y]
        if not len(d): continue
        rel = pd.DataFrame({t: ind[t]["close"].reindex(d).ffill() for t in uni})
        rel = rel / rel.bfill().iloc[0]
        path = val * rel.mean(axis=1) * (1 - COST_BUY)
        vals.append(path); val = path.iloc[-1] * (1 - COST_SELL)
    return pd.concat(vals)

## 5. 성과 지표 — 수익 · 위험 · 효율 · 실전 가능성

In [ ]:
km = kospi.loc[START:END]

def metrics(eq, trades=None):
    r = eq.pct_change().dropna()
    yrs = len(r) / 252
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1/yrs) - 1
    vol = r.std() * np.sqrt(252)
    mdd = (eq / eq.cummax() - 1).min()
    rm = km.pct_change().reindex(r.index).fillna(0)
    beta = np.cov(r, rm)[0, 1] / rm.var()
    alpha = (r - beta * rm).mean() * 252
    r10 = eq.pct_change(10).dropna()                 # 약 15일(10거래일) 수익률
    out = {"CAGR": cagr, "Vol": vol, "Sharpe": (cagr - RF) / vol, "MDD": mdd,
           "Calmar": cagr / abs(mdd) if mdd else np.nan,
           "VaR95_15d": r10.quantile(0.05), "Beta": beta, "Alpha": alpha}
    if trades is not None and len(trades):
        win = trades["ret"] > 0
        avg_w, avg_l = trades.loc[win, "ret"].mean(), -trades.loc[~win, "ret"].mean()
        streak = (~win).astype(int).groupby(win.cumsum()).sum().max()
        out.update({"Trades": len(trades), "WinRate": win.mean(),
                    "Payoff": avg_w / avg_l if avg_l else np.nan,
                    "Expectancy": win.mean() * avg_w - (1 - win.mean()) * avg_l,
                    "MaxLossStreak": streak, "AvgHoldDays": trades["days"].mean()})
    return pd.Series(out)

ORDER = ["CAGR", "Alpha", "Vol", "MDD", "VaR95_15d", "Beta", "Sharpe", "Calmar",
         "Trades", "WinRate", "Payoff", "Expectancy", "MaxLossStreak", "AvgHoldDays"]

def fmt(t):
    t = t[[c for c in ORDER if c in t]]
    pct = ["CAGR", "Vol", "MDD", "VaR95_15d", "Alpha", "WinRate", "Expectancy"]
    num = ["Sharpe", "Calmar", "Beta", "Payoff", "AvgHoldDays"]
    return (t.style.format("{:.1%}", subset=[c for c in pct if c in t], na_rep="-")
             .format("{:.2f}", subset=[c for c in num if c in t], na_rep="-")
             .format("{:.0f}", subset=[c for c in ["Trades", "MaxLossStreak"] if c in t], na_rep="-"))

## 6. 실행 — 보험주 / 비보험주 각각 L0~L4

In [ ]:
curves, tables, logs = {}, {}, {}
for group, max_pos in [("Ins", 5), ("NonIns", 10)]:
    rows = {}
    eq0 = run_hold(group); curves[(group, "L0")] = eq0; rows["L0 Hold"] = metrics(eq0)
    for layer in ["L1", "L2", "L3", "L4"]:
        eq, tr = run(group, layer, max_pos)
        curves[(group, layer)], logs[(group, layer)] = eq, tr
        rows[layer] = metrics(eq, tr)
        tr.to_csv(f"results/trades_{group}_{layer}.csv", index=False)
    kospi_eq = km / km.iloc[0]
    rows["KOSPI"] = metrics(kospi_eq)
    tables[group] = pd.DataFrame(rows).T
    print(f"\n===== {group} (최대 {max_pos}종목) =====")
    display(fmt(tables[group]))

## 7. 그래프 (빨간 점선 = 선거·정치 이벤트)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
for ax, group in zip(axes, ["Ins", "NonIns"]):
    for layer in ["L0", "L1", "L2", "L3", "L4"]:
        curves[(group, layer)].plot(ax=ax, label=layer, logy=True)
    (km / km.iloc[0]).plot(ax=ax, label="KOSPI", color="black", ls="--", logy=True)
    for r in events.itertuples():
        if r.category in ("election", "politics"):
            ax.axvline(r.date, color="red", ls=":", alpha=.4, lw=1)
    ax.set_title(group); ax.legend(ncol=6); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. 매매 기록 살펴보기 — 청산 이유별 성과

In [ ]:
for key in [("NonIns", "L2"), ("NonIns", "L4")]:
    tr = logs[key]
    if len(tr):
        print(key)
        display(tr.groupby("reason")["ret"].agg(["count", "mean", "median"]).style.format("{:.1%}", subset=["mean", "median"]))

## 9. 해석 메모 → experiments.md

- **L1 vs L0**: 추세 타이밍이 그냥 보유보다 나았나? (특히 MDD)
- **L2 vs L1**: 거래량·OBV 확인이 승률/손익비를 올렸나, 아니면 기회만 줄였나?
- **L3 vs L2**: 볼린저 상단 근처 진입(강세 확인)이 도움이 됐나? → 반대로 `BB_POS_MIN`을 낮춰보기
- **L4 vs L2**: 손절·비중조절로 MDD·최대연속손실이 얼마나 줄었고, 수익은 얼마나 포기했나?
- **보험주 vs 비보험주**: 추세추종이 변동성 작은 보험주에서도 통하나?
- 추세추종(B 전략)답게 **승률 낮고 손익비 높게** 나오는지 확인

**바꿔볼 것:** MACD (12,26,9) → (5,20,5) / MONTH_MA 10 → 6 / ATR_MULT 2 → 3 / 판단 주기 15일 → 매일
